In [1]:
import pandas as pd
import numpy as np
from math import log
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.preprocessing import MultiLabelBinarizer
import statsmodels.api as sm
import re
import os
from matplotlib.patches import Patch
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde


In [ ]:
def is_all_equal(labels):
    unique_labels = set(labels)
    return 1 if len(unique_labels) == 1 else 0

def is_all_equal(labels):
    unique_labels = set(labels)
    return 1 if len(unique_labels) == 1 else 0

def compute_metrics(df):
    df_len_var_step1 = (
        df
        .groupby(["benchmark", "prompt_id", "model", "question_id"])
        .agg(output_len_var=("output_len", "var"))
        .reset_index()
    )

    df_len_var = (
        df_len_var_step1
        .groupby(["benchmark", "model", "prompt_id"])
        .agg(output_tokens_var=("output_len_var", "mean"))
        .reset_index()
    )

    df_acc_var_len = (
        df
        .groupby(["benchmark", "prompt_id", "question_lang", "model"])
        .mean(numeric_only=True)
        .reset_index()
        .groupby(["benchmark", "model", "prompt_id"])
        .agg(
            acc_mean=("is_correct", "mean"),
            acc_var=("is_correct", "var"),
        )
        .reset_index()
    )
    df_consistency = (
        df
        .groupby(["benchmark", "model", "prompt_id", "question_id"])["pred_label"]
        .apply(is_all_equal)
        .reset_index(name="all_equal_consistency")
        .groupby(["benchmark", "model", "prompt_id"])["all_equal_consistency"]
        .mean()
        .reset_index(name="consistency")
    )

    df_merged = pd.merge(df_acc_var_len, df_consistency, on=["benchmark", "model", "prompt_id"], how="left")
    df_merged = pd.merge(df_merged, df_len_var, on=["benchmark", "model", "prompt_id"], how="left")
    return df_merged

In [4]:
#RQ1-1 data processing
file_path = "/shared/4/projects/llm-personas/tmp/experiment1_simple.jsonl"
df_exp1 = pd.read_json(file_path, lines=True)
df_exp1_samelang = df_exp1[df_exp1["prompt_lang"] == df_exp1["question_lang"]]
df_exp1_crosslang = df_exp1[df_exp1["prompt_lang"] == "en"]
exp1_samelang_metric = compute_metrics(df_exp1_samelang)
exp1_crosslang_metric = compute_metrics(df_exp1_crosslang)
exp1_samelang_metric = compute_metrics(df_exp1_samelang)
exp1_crosslang_metric = compute_metrics(df_exp1_crosslang)

In [ ]:
# RQ1-1 plotting
def plot_grouped_metrics_final(df1, df2, label1="Same Language", label2="English Prompt", save_path="figures/exp1-1_same_lang_vs_en.pdf"):
    plt.rcParams.update({
        'font.size': 26,
        'axes.titlesize': 26,
        'axes.labelsize':  26,
        'xtick.labelsize': 26,
        'ytick.labelsize': 26,
        'legend.fontsize': 26
    })

    sns.set_style("whitegrid")

    metrics = ["acc_mean", "acc_var", "consistency", "output_tokens_var"]
    metric_labels = {
    "acc_mean": "Accuracy Mean (↑)",
    "acc_var": "Accuracy Variance (↓)",
    "consistency": "Consistency (↑)",
    "output_tokens_var": "Length Variance (↓)"
}

    df1_long = df1[metrics].melt(var_name="metric", value_name="value")
    df1_long["label"] = label1

    df2_long = df2[metrics].melt(var_name="metric", value_name="value")
    df2_long["label"] = label2

    df_plot = pd.concat([df1_long, df2_long], ignore_index=True)

    means = df_plot.groupby(["label", "metric"])["value"].mean().reset_index()
    print("\n===== Mean values used for plotting =====")
    for metric in metrics:
        print(f"\n### {metric_labels[metric]} ({metric})")
        for _, row in means[means["metric"] == metric].iterrows():
            val = row["value"]
            if metric == "output_tokens_var":
                print(f"{row['label']:<20}: {int(val)}")
            else:
                print(f"{row['label']:<20}: {val:.4f}")
    print("==========================================\n")

    fig, axes = plt.subplots(1, 4, figsize=(16, 10), sharey=False)
    palette = sns.color_palette("tab10")

    for i, metric in enumerate(metrics):
        ax = axes[i]
        plot_data = df_plot[df_plot["metric"] == metric].copy()

        if metric == "output_tokens_var":
            plot_data["value"] = plot_data["value"] / 1e5

        sns.barplot(data=plot_data, x="metric", y="value", hue="label", ax=ax, palette=palette, width=0.6)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xticks([])
        ax.set_xticklabels([])
        ax.set_title('', pad=10)
        ax.text(0.5, -0.08, metric_labels[metric], ha='center', va='top',
                transform=ax.transAxes, fontsize=24)

        for j, bar in enumerate(ax.patches):
            height = bar.get_height()
            if height == 0:  
                continue
            xpos = bar.get_x() + bar.get_width() / 2
            ymax = ax.get_ylim()[1]
            if j % 2 == 0:
                xpos -= 0.05 
            if metric == "output_tokens_var":
                text = f"{height:.2f}"
            else:
                text = f"{int(height):,}" if height >= 10000 else f"{height:.3f}"
            ax.text(xpos, height + 0.03 * ymax, text, ha='center', va='bottom', fontsize=18)

        ax.axhline(0, color='black', linewidth=1)
        current_ylim = ax.get_ylim()
        ax.set_ylim(top=current_ylim[1] * 1.05)
        ax.get_legend().remove()
        if metric == "output_tokens_var":
            ax.text(0.1, 1, r"$\times 10^5$", transform=ax.transAxes,
                    ha='center', va='bottom', fontsize=16, style='italic')

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.98), fontsize=26)

    plt.tight_layout(rect=[0, 0, 1, 0.9])
    plt.savefig(save_path, format='pdf', dpi=300, bbox_inches='tight')
    plt.show()

plot_grouped_metrics_final(exp1_samelang_metric, exp1_crosslang_metric, label1="Same Language", label2="English Prompt")


In [ ]:
#RQ1-2 data processing
exp1_2_metric = exp1_samelang_metric.groupby(["model", "prompt_id"])[["acc_mean", "acc_var", "consistency", "output_tokens_var"]].mean().reset_index()
exp1_2_baseline = pd.read_json("/shared/4/projects/llm-personas/tmp/exp1_baseline.jsonl", lines=True)
exp1_2_baseline_metric = compute_metrics(exp1_2_baseline)
exp1_2_baseline_metric = exp1_2_baseline_metric.groupby(["model", "prompt_id"])[["acc_mean", "acc_var", "consistency", "output_tokens_var"]].mean().reset_index()

In [ ]:
#RQ1-2 plotting
plt.rcParams.update(plt.rcParamsDefault)
data = exp1_2_metric.rename(columns={"output_tokens_var": "len_var"})
baseline = exp1_2_baseline_metric.rename(columns={"output_tokens_var": "len_var"})

value_list = ["acc_mean", "acc_var", "consistency", "len_var"]
label_list = [
    "Mean Accuracy Across Languages",
    "Variance of Accuracy Across Languages",
    "Consistency of Model Output Across Language",
    "Variance of Output Length Across Languages"
]

def format_label_latex(label):
    if "_" in label:
        parts = label.split("_", 1)
        return rf"$\mathrm{{{parts[0].capitalize()}}}_{{\mathrm{{{parts[1]}}}}}$"
    else:
        return rf"$\mathrm{{{label.capitalize()}}}$"

palette = sns.color_palette("tab10")
model_palette = dict(zip(data["model"].unique(), palette))

prompt_marker_label = {
    "p_1000": ("X", "Empty Prompt"),
    "p_1001": ("*", "Think Step by Step")
}

y_value = value_list[0]

for idx, variable_1 in enumerate([1, 2, 3]):
    x_value = value_list[variable_1]

    plt.figure(figsize=(9, 6))
    ax = plt.gca()

    x_label = f"{label_list[variable_1]}"
    y_label = f"{label_list[0]}"

    sns.scatterplot(
        data=data,
        x=x_value,
        y=y_value,
        hue="model",
        s=20,
        palette=model_palette,
        legend=False,
        ax=ax
    )

    for _, row in baseline.iterrows():
        prompt_id = row["prompt_id"]
        model_name = row["model"]
        if prompt_id in prompt_marker_label:
            marker, _ = prompt_marker_label[prompt_id]
            color = model_palette.get(model_name, "gray")
            ax.scatter(
                row[x_value],
                row[y_value],
                marker=marker,
                facecolor=color,
                edgecolor="black",
                s=100,
                label=None
            )

    ax.set_xlabel(x_label, fontsize=20)
    ax.set_ylabel(y_label, fontsize=20)
    ax.grid(True)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0.2)
    ax.tick_params(labelsize=20)

    unique_models = data["model"].unique()
    custom_handles = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=model_palette[m], markersize=7, label=m)
        for m in unique_models
    ]
    custom_handles += [
        Line2D([0], [0], marker="X", color="gray", markerfacecolor="gray",
               markeredgecolor="black", markersize=8, linestyle='None', label="Empty Prompt"),
        Line2D([0], [0], marker="*", color="gray", markerfacecolor="gray",
               markeredgecolor="black", markersize=8, linestyle='None', label="Think Step by Step"),
    ]
    ax.legend(
        handles=custom_handles,
        title=None,
        fontsize=18,
        loc='best',
        frameon=True
    )

    plt.tight_layout()
    plt.savefig(f"figures/exp1_2_{x_value}.pdf", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


In [ ]:
#RQ1-3 data processing
file_path = "/shared/4/projects/llm-personas/tmp/experiment1_simple.jsonl"
sys_prompt_df = pd.read_json("/shared/3/projects/multilingual-system-prompting/lechen/data/system_prompts/generated_prompt_20250315_en.jsonl", lines=True)
sys_prompt_df["prompt_id"] = np.array(range(len(sys_prompt_df)))
sys_prompt_df["prompt_id"] = sys_prompt_df["prompt_id"].apply(lambda x: "p_"+str(x))

exp1_3_samelang_metric = exp1_samelang_metric.copy()
exp1_3_crosslang_metric = exp1_crosslang_metric.copy()
exp1_3_samelang_metric["is_cross_language"] = 0
exp1_3_crosslang_metric["is_cross_language"] = 1
exp1_3_combined_metric = pd.concat([exp1_3_samelang_metric, exp1_3_crosslang_metric], ignore_index=True)
exp1_3_combined_metric_final = exp1_3_combined_metric.groupby(["prompt_id", "is_cross_language"], as_index=False).mean(numeric_only=True)
exp1_3_combined_metric_final = exp1_3_combined_metric_final.merge(sys_prompt_df, on="prompt_id")
exp1_3_combined_metric_final = exp1_3_combined_metric_final.rename(columns={"output_tokens_var": "len_var"})

In [ ]:
# RQ1-3 plotting
def plot_regression_heatmap_transposed(
    df_with_prompt, 
    target_list=["acc_mean", "acc_var", "consistency", "len_var"],
    save_csv_path="figures/exp1_regression_results.csv"
):
    plt.rcParams["text.usetex"] = False

    target_label_map = {
        "acc_mean": r"$\mathrm{acc}_{\mathrm{mean}}$",
        "acc_var": r"$1 - \mathrm{acc}_{\mathrm{var}}$",
        "consistency": r"$\mathrm{consistency}$",
        "len_var": r"$1 - \mathrm{len}_{\mathrm{var}}$"
    }

    transform_targets = {"acc_var", "len_var"}

    mlb = MultiLabelBinarizer()
    df = df_with_prompt.copy()
    X_cat_all = pd.DataFrame(
        mlb.fit_transform(df["category"]), 
        columns=mlb.classes_
    )
    df_encoded = pd.concat(
        [df.drop(columns="category").reset_index(drop=True), X_cat_all], 
        axis=1
    )

    all_components = mlb.classes_.tolist()
    rows = []

    for target in target_list:
        y_raw = df_encoded[target]
        if target in transform_targets:
            y_raw = 1 - y_raw
        y = (y_raw - y_raw.min()) / (y_raw.max() - y_raw.min())

        X = df_encoded[all_components]
        X = X.loc[:, X.sum() > 0]
        X_with_const = sm.add_constant(X)
        model_fit = sm.OLS(y, X_with_const).fit()

        for comp in X.columns:
            coef = model_fit.params.get(comp, float('nan'))
            pval = model_fit.pvalues.get(comp, float('nan'))
            rows.append({
                "target": target,
                "component": comp,
                "coef": coef,
                "pval": pval
            })

    result_df = pd.DataFrame(rows)
    result_df.to_csv(save_csv_path, index=False)

    def get_significance_marker(p):
        if p < 0.001: return '***'
        elif p < 0.01: return '**'
        elif p < 0.05: return '*'
        else: return ''

    result_df['annot'] = result_df['pval'].map(get_significance_marker)
    result_df['target_latex'] = result_df['target'].map(target_label_map)

    coef_matrix = result_df.pivot(index='target_latex', columns='component', values='coef')
    annot_matrix = result_df.pivot(index='target_latex', columns='component', values='annot')

    latex_target_order = [target_label_map[t] for t in target_list]
    coef_matrix = coef_matrix.reindex(latex_target_order)
    annot_matrix = annot_matrix.reindex(latex_target_order)

    plt.figure(figsize=(10, 3))
    sns.heatmap(
        coef_matrix.astype(float), 
        annot=annot_matrix, 
        fmt="", 
        cmap="coolwarm", 
        center=0,
        cbar_kws={'label': 'Regression Coef'}
    )
    plt.ylabel("Evaluation Metric")
    plt.xlabel("Component")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(f"figures/exp1_regression.pdf")
    plt.show()

plot_regression_heatmap_transposed(
    exp1_3_combined_metric_final,
    target_list=["acc_mean", "acc_var", "consistency", "len_var"]
)


In [ ]:
#RQ3-1 data processing
df_exp1 = pd.read_json("/home/yszhou/multilingual_project/experiment/exp3_evaluation/sentence_classifier_new/result/0723_lang_detections_result.jsonl", lines=True)
df_exp2 = pd.read_json("/home/yszhou/multilingual_project/experiment/exp3_evaluation/sentence_classifier_new/result/exp2_lang_detections_result.jsonl", lines=True)
df_exp2.rename(columns={"system_prompt": "prompt_id"}, inplace=True)
def process_detected_langs(df, keep_langs=['en', 'zh', 'es', 'fr', 'hi']):
    df_lang_expanded = df["detected_langs_count"].apply(pd.Series).fillna(0)
    df_lang_expanded = df_lang_expanded.astype(int)
    df_expanded = pd.concat([df, df_lang_expanded], axis=1)
    group_cols = ["benchmark", "prompt_id", "question_lang"]
    lang_cols = df_lang_expanded.columns.tolist()
    df_grouped_mean = df_expanded.groupby(group_cols)[lang_cols].mean().reset_index()
    df_grouped_question_lang = df_grouped_mean.groupby("question_lang").mean(numeric_only=True).reset_index()
    all_numeric_cols = df_grouped_question_lang.select_dtypes(include="number").columns
    other_langs = [col for col in all_numeric_cols if col not in keep_langs]
    df_grouped_question_lang["other"] = df_grouped_question_lang[other_langs].sum(axis=1)
    all_numeric_cols = df_grouped_question_lang.select_dtypes(include="number").columns
    group_cols = [col for col in df_grouped_question_lang.columns if col not in all_numeric_cols]
    df_grouped_question_lang = df_grouped_question_lang[group_cols + keep_langs + ["other"]]
    return df_grouped_question_lang
df_exp1_summary = process_detected_langs(df_exp1)
df_exp2_summary = process_detected_langs(df_exp2)

In [ ]:
#RQ3-1 plotting
def plot_language_distribution_stacked(df1, df2, save_path=None):
    import os
    group_col = "q_lang"
    df1 = df1.rename(columns={"question_lang": group_col})
    df2 = df2.rename(columns={"question_lang": group_col})

    def preprocess(df, version_label):
        df_melted = df.melt(id_vars=group_col, var_name="output_lang", value_name="count")

        def map_lang(row):
            if row[group_col] == "en" and row["output_lang"] == "en":
                return "question_lang"
            elif row["output_lang"] == "en":
                return "en"
            elif row["output_lang"] == row[group_col]:
                return "question_lang"
            else:
                return "other"

        df_melted["lang_group"] = df_melted.apply(map_lang, axis=1)
        df_grouped = (
            df_melted.groupby([group_col, "lang_group"])["count"]
            .sum()
            .reset_index()
        )
        df_grouped["version"] = version_label
        return df_grouped

    df1_processed = preprocess(df1, "Before")
    df2_processed = preprocess(df2, "After")
    df_combined = pd.concat([df1_processed, df2_processed], ignore_index=True)

    df_combined["lang_group"] = pd.Categorical(
        df_combined["lang_group"], categories=["question_lang", "en", "other"], ordered=True
    )

    version_order = df_combined["version"].unique().tolist()
    df_combined["version"] = pd.Categorical(df_combined["version"], categories=version_order, ordered=True)

    df_pivot = df_combined.pivot_table(
        index=[group_col, "version"],
        columns="lang_group",
        values="count",
        fill_value=0,
        aggfunc='sum'
    ).copy()

    df_abs_for_csv = df_pivot.reset_index().copy()

    df_pivot["total"] = df_pivot.sum(axis=1)
    for col in ["question_lang", "en", "other"]:
        df_pivot[col] = df_pivot[col] / df_pivot["total"] * 100
    df_pivot.drop(columns="total", inplace=True)

    df_prop_for_csv = df_pivot.reset_index().copy()

    if save_path:
        base = os.path.splitext(save_path)[0]
        abs_path = f"{base}_absolute.csv"
        prop_path = f"{base}_proportion.csv"
    else:
        abs_path = "language_distribution_absolute.csv"
        prop_path = "language_distribution_proportion.csv"

    df_abs_for_csv.to_csv(abs_path, index=False)
    df_prop_for_csv.to_csv(prop_path, index=False)

    df_pivot = df_pivot.reset_index()
    df_pivot["label"] = df_pivot[group_col].astype(str) + " - " + df_pivot["version"].astype(str)

    lang_order = ["en", "es", "fr", "hi", "zh"]
    row_order = []
    for lang in lang_order:
        for version in version_order:
            row_order.append(f"{lang} - {version}")
        row_order.append(f"gap-{lang}")

    df_pivot["label"] = pd.Categorical(df_pivot["label"], categories=row_order, ordered=True)
    df_pivot = df_pivot.set_index("label").sort_index()

    zero_row = {col: 0.0 for col in df_pivot.columns}
    for gap_label in [f"gap-{lang}" for lang in lang_order]:
        df_pivot.loc[gap_label] = zero_row
    df_pivot = df_pivot.loc[row_order]

    color_map = {
        "question_lang": "#55A868",
        "en": "#4C72B0",
        "other": "#EFC75E",
    }
    fontsize = 20
    fig, ax = plt.subplots(figsize=(13, 8))
    left = [0] * len(df_pivot)
    bar_height = 0.7

    for col in ["question_lang", "en", "other"]:
        bar_colors = [color_map[col] if not str(idx).startswith("gap-") else "none" for idx in df_pivot.index]
        edge_colors = ["black" if not str(idx).startswith("gap-") else "none" for idx in df_pivot.index]
        ax.barh(df_pivot.index, df_pivot[col], left=left, height=bar_height,
                color=bar_colors, edgecolor=edge_colors, label=col)
        left = [l + v for l, v in zip(left, df_pivot[col])]

    ax.set_xlim(0, 100)
    ax.set_xlabel("Proportion (%)", fontsize=fontsize)
    ax.set_ylabel("Task Language -- Prompt Type", fontsize=fontsize)

    lang_fullname_map = {
        "en": "English", "es": "Spanish", "fr": "French", "hi": "Hindi", "zh": "Chinese"
    }

    def convert_label(label):
        if str(label).startswith("gap-"):
            return ""
        lang_code, version = str(label).split(" - ")
        version_map = {
            "Before": "Random",
            "After": "Optimized",
            "Random": "Random",
            "Optimized": "Optimized"
        }
        return f"{lang_fullname_map.get(lang_code, lang_code)} - {version_map.get(version, version)}"

    yticklabels = [convert_label(label) for label in df_pivot.index]
    ax.set_yticks(range(len(df_pivot)))
    ax.set_yticklabels(yticklabels, fontsize=fontsize)

    ax.tick_params(axis='x', labelsize=fontsize)
    ax.invert_yaxis()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
    plt.show()


plot_language_distribution_stacked(df_exp1_summary, df_exp2_summary, save_path="/home/yszhou/multilingual_project/experiment/analysis/figures/exp3_response_lang_shift.pdf")

In [ ]:
#RQ3-2 data processing
df_exp1_with_language = pd.read_json("/home/yszhou/multilingual_project/experiment/exp3_evaluation/sentence_classifier_new/result/exp1_result_with_language.jsonl", lines=True)
df_exp1_with_language.rename(columns={"Logistical Reasoning": "Logical Reasoning"}, inplace=True)
df_exp1_with_language_math500 = df_exp1_with_language[df_exp1_with_language["benchmark"] == "math500"]
df_exp1_with_language_mmlupro = df_exp1_with_language[df_exp1_with_language["benchmark"] == "mmlupro"]
df_exp1_with_language_unimoral = df_exp1_with_language[df_exp1_with_language["benchmark"] == "unimoral"]

def get_reasoning_distribution_df(df):
    predefined_type_map = {
        "1": "Retrieval",
        "2": "Reframing",
        "3": "Logical Reasoning",
        "4": "Calculation",
        "5": "Subgoal setting",
        "6": "Backtracking",
        "7": "Verification",
        "8": "Backward chaining",
        "9": "Others"  
    }

    reason_id_to_name = {int(k): v for k, v in predefined_type_map.items()}
    target_langs = {"en", "es", "fr", "hi", "zh"}
    reason_names = list(reason_id_to_name.values())
    counts = {lang: {name: 0 for name in reason_names} for lang in target_langs}

    for _, row in df.iterrows():
        langs = row["language_detection"]
        reasons = row["reasoning_vector"]
        flat_langs = [l[0] for l in langs if isinstance(l, list) and l and isinstance(l[0], str)]
        valid_langs = [l for l in flat_langs if l in target_langs]
        if not valid_langs:
            continue
        for lang in valid_langs:
            for r in reasons:
                if r in reason_id_to_name:
                    counts[lang][reason_id_to_name[r]] += 1

    df_counts = pd.DataFrame(counts).T.fillna(0).astype(int)
    df_counts.index.name = "Language"
    return df_counts
df_exp1_with_language_math500_count = get_reasoning_distribution_df(df_exp1_with_language_math500)
df_exp1_with_language_mmlupro_count = get_reasoning_distribution_df(df_exp1_with_language_mmlupro)
df_exp1_with_language_unimoral_count = get_reasoning_distribution_df(df_exp1_with_language_unimoral)

In [ ]:
#RQ3-2 plotting
def plot_reasoning_distribution_multiple(dfs, title):
    import os
    df_concat = pd.concat(dfs)
    df_grouped = df_concat.groupby("Language").mean(numeric_only=True)
    df_ratio = df_grouped.div(df_grouped.sum(axis=1), axis=0).reset_index()
    df_melted = df_ratio.melt(id_vars="Language", var_name="Reasoning Type", value_name="Proportion")

    out_dir = "/home/yszhou/multilingual_project/experiment/analysis/figures_1201"
    base = os.path.join(out_dir, title)
    df_grouped.reset_index().to_csv(f"{base}_absolute.csv", index=False)
    df_ratio.to_csv(f"{base}_proportion_wide.csv", index=False)
    df_melted.to_csv(f"{base}_proportion_long.csv", index=False)

    lang_map = {
        "en": "English",
        "es": "Spanish",
        "hi": "Hindi",
        "zh": "Chinese",
        "fr": "French"
    }
    df_melted["Language"] = df_melted["Language"].map(lang_map)

    fontsize = 30
    plt.figure(figsize=(14, 8))
    ax = sns.barplot(data=df_melted, x="Language", y="Proportion", hue="Reasoning Type")
    plt.ylabel("Proportion", fontsize=fontsize)
    plt.xlabel("Language", fontsize=fontsize)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles, labels=labels, title="Reasoning Behavior",
              loc="lower center", bbox_to_anchor=(0.5, 1.05),
              ncol=3, fontsize=fontsize - 7, title_fontsize=fontsize - 2, frameon=False)
    plt.tight_layout()
    plt.savefig(f"/home/yszhou/multilingual_project/experiment/analysis/figures_1201/{title}.pdf")
    plt.show()

plot_reasoning_distribution_multiple(
    [df_exp1_with_language_mmlupro_count, df_exp1_with_language_math500_count, df_exp1_with_language_unimoral_count],
    title="exp3_behaviors_proportion_language"
)


In [ ]:
#RQ3-3 data processing
def data_process(df_all, col_name="output"):
    predefined_type_map = {
        "1": "Retrieval",
        "2": "Reframing",
        "3": "Logistical Reasoning",
        "4": "Calculation",
        "5": "Subgoal setting",
        "6": "Backtracking",
        "7": "Verification",
        "8": "Backward chaining",
        "9": "Others"  
    }

    type_name_to_id = {v.lower(): int(k) for k, v in predefined_type_map.items()}

    def extract_final_answer(text):
        if not isinstance(text, str):
            return ""
        marker = "[Final answer]"
        idx = text.lower().rfind(marker.lower())
        if idx == -1:
            return ""
        return text[idx + len(marker):].strip()

    def extract_step_labels(text):
        if not isinstance(text, str):
            return []
        label_ids = []
        step_contents = re.findall(r"<step_\d+>([^\n]*)", text)
        for step in step_contents:
            step_lower = step.lower()
            matched = None

            for name, idx in type_name_to_id.items():
                if name in step_lower:
                    matched = idx
                    break

            if matched is None:
                type_match = re.search(r"<type_name_(\d+)>", step_lower)
                if type_match:
                    x = int(type_match.group(1))
                    if 1 <= x <= 9:
                        matched = x

            if matched is not None:
                label_ids.append(matched)
            else:
                label_ids.append(9)
        return label_ids
    df_all["final_answer"] = df_all[col_name].apply(extract_final_answer)
    df_all["reasoning_vector"] = df_all["final_answer"].apply(extract_step_labels)
    return df_all

df_exp1_result = pd.read_json("/home/yszhou/multilingual_project/experiment/exp3_evaluation/sentence_classifier_new/result/combined_result_simplified.jsonl", lines=True)
sys_prompt_df = pd.read_json("/home/yszhou/multilingual_project/experiment/exp3_evaluation/sentence_classifier_new/analysis/generated_prompt_20250315_en.jsonl", lines=True)
sys_prompt_df["prompt_id"] = np.array(range(len(sys_prompt_df)))
sys_prompt_df["prompt_id"] = sys_prompt_df["prompt_id"].apply(lambda x: "p_"+str(x))
df_exp1_result = df_exp1_result.merge(sys_prompt_df, on="prompt_id", how="left")
predefined_type_map = {
    "1": "Retrieval",
    "2": "Reframing",
    "3": "Logical Reasoning",
    "4": "Calculation",
    "5": "Subgoal setting",
    "6": "Backtracking",
    "7": "Verification",
    "8": "Backward chaining",
    "9": "Others"  
}

reasoning_cols = [
    "Backtracking", "Backward chaining", "Retrieval", "Verification",
    "Subgoal setting", "Reframing", "Others", "Calculation",
    "Logical Reasoning"
]

def count_types(vec):
    vec = list(map(str, vec))
    count_dict = {v: 0 for v in reasoning_cols}
    for code in vec:
        name = predefined_type_map.get(code)
        if name in count_dict:
            count_dict[name] += 1
    return pd.Series(count_dict)

df_type_counts = df_exp1_result["reasoning_vector"].apply(count_types)
df_exp1_result = pd.concat([df_exp1_result, df_type_counts], axis=1)
reasoning_cols = [
    "Backtracking", "Backward chaining", "Retrieval", "Verification",
    "Subgoal setting", "Reframing", "Others", "Calculation",
    "Logical Reasoning"
]

df_exp1_grouped = df_exp1_result.groupby(["prompt", "benchmark"])[reasoning_cols].mean().reset_index()
df_exp1_acc = df_exp1_result.groupby(["prompt", "benchmark"])["is_correct"].mean().reset_index(name="avg_acc")
df_exp1_grouped = df_exp1_grouped.merge(df_exp1_acc, on=["prompt", "benchmark"])
df_exp1_grouped["type"] = "exp1"

file_path = "/shared/4/projects/llm-personas/tmp/exp2/qwen/exp2_combined.jsonl"
df_exp2 = pd.read_json(file_path, lines=True)
df_exp2_result = data_process(df_exp2, col_name="output_judging")
predefined_type_map = {
    "1": "Retrieval",
    "2": "Reframing",
    "3": "Logical Reasoning",
    "4": "Calculation",
    "5": "Subgoal setting",
    "6": "Backtracking",
    "7": "Verification",
    "8": "Backward chaining",
    "9": "Others"  
}

def count_types(vec):
    vec = list(map(str, vec))
    counts = {name: vec.count(code) for code, name in predefined_type_map.items()}
    return pd.Series(counts)

df_type_counts = df_exp2_result["reasoning_vector"].apply(count_types)
df_exp2_result = pd.concat([df_exp2_result, df_type_counts], axis=1)
reasoning_cols = [
    "Backtracking", "Backward chaining", "Retrieval", "Verification",
    "Subgoal setting", "Reframing", "Others", "Calculation",
    "Logical Reasoning"
]

df_exp2_grouped = df_exp2_result.groupby(["system_prompt", "benchmark"])[reasoning_cols].mean().reset_index()
df_exp2_acc = df_exp2_result.groupby(["system_prompt", "benchmark"])["is_correct"].mean().reset_index(name="avg_acc")
df_exp2_grouped = df_exp2_grouped.merge(df_exp2_acc, on=["system_prompt", "benchmark"])
df_exp2_grouped.rename(columns={"system_prompt": "prompt"}, inplace=True)
df_exp2_grouped["type"] = "exp2"

df_combined = pd.concat([df_exp1_grouped, df_exp2_grouped], ignore_index=True)
df_combined["benchmark"] = df_combined["benchmark"].replace("mmlu_pro", "mmlupro")

In [ ]:
#RQ3-3 plotting
def plot_lollipop(df_combined, save_path="/home/yszhou/multilingual_project/experiment/analysis/figures_1201/exp3_lollipop_counting.pdf"):
    import os
    reasoning_cols = [
        "Backtracking", "Backward chaining", "Retrieval", "Verification",
        "Subgoal setting", "Reframing", "Others", "Calculation", "Logical Reasoning"
    ]
    df = df_combined.copy()
    df["type"] = df["type"].replace({"exp1": "Randomized", "exp2": "Optimized"})
    df_grouped = df.groupby(["benchmark", "type"])[reasoning_cols].mean().reset_index()
    df_melted = df_grouped.melt(id_vars=["benchmark", "type"], value_vars=reasoning_cols,
                                var_name="Reasoning Type", value_name="Average Count")
    order_before = (df_melted[df_melted["type"] == "Randomized"]
                    .groupby("Reasoning Type")["Average Count"].mean()
                    .sort_values(ascending=False).index.tolist())
    df_melted["Reasoning Type"] = pd.Categorical(df_melted["Reasoning Type"], categories=order_before, ordered=True)

    base = os.path.splitext(save_path)[0]
    csv_path = f"{base}_plotdata.csv"
    df_melted.to_csv(csv_path, index=False)

    benchmark_colors = {
        "mmlupro": (202/255, 0/255, 32/255),
        "math500": (0/255, 136/255, 55/255),
        "unimoral": (5/255, 113/255, 176/255)
    }
    type_markers = {"Randomized": "o", "Optimized": "*"}
    plt.figure(figsize=(18, 10))
    sns.set(style="whitegrid")
    reasoning_types = df_melted["Reasoning Type"].cat.categories.tolist()
    x = np.arange(len(reasoning_types))
    benches = list(df_melted["benchmark"].unique())
    point_width = 0.18
    s_size = 540
    added_labels = set()

    for i, bench in enumerate(benches):
        color = benchmark_colors[bench]
        for j, rt in enumerate(reasoning_types):
            y_b = df_melted[(df_melted["benchmark"] == bench) & (df_melted["type"] == "Randomized") & (df_melted["Reasoning Type"] == rt)]["Average Count"].values
            y_a = df_melted[(df_melted["benchmark"] == bench) & (df_melted["type"] == "Optimized") & (df_melted["Reasoning Type"] == rt)]["Average Count"].values
            if len(y_b) == 0 or len(y_a) == 0:
                continue
            xi = x[j] + (i - len(benches)/2) * point_width + point_width/2
            y0, y1 = y_b[0], y_a[0]
            if abs(y0 - y1) < 0.01:
                y0 -= 0.006
                y1 += 0.006
            plt.plot([xi, xi], [y0, y1], color="black", linewidth=3, zorder=2)
            label_b = f"Randomized-{bench}"
            label_a = f"Optimized-{bench}"
            if label_b not in added_labels:
                plt.scatter(xi, y0, color=color, marker=type_markers["Randomized"], edgecolors="black", s=s_size, zorder=3, label=label_b)
                added_labels.add(label_b)
            else:
                plt.scatter(xi, y0, color=color, marker=type_markers["Randomized"], edgecolors="black", s=s_size, zorder=3)
            if label_a not in added_labels:
                plt.scatter(xi, y1, color=color, marker=type_markers["Optimized"], edgecolors="black", s=s_size+80, zorder=3, label=label_a)
                added_labels.add(label_a)
            else:
                plt.scatter(xi, y1, color=color, marker=type_markers["Optimized"], edgecolors="black", s=s_size+80, zorder=3)

    font_size = 28
    plt.xticks(x, reasoning_types, rotation=30, ha="right", fontsize=font_size-4)
    plt.yticks(fontsize=font_size)
    plt.xlabel("Reasoning Behavior", fontsize=font_size-2)
    plt.ylabel("Average Count", fontsize=font_size)
    plt.legend(title="Prompt Type - Benchmark", title_fontsize=font_size-6, fontsize=font_size-4, loc="upper right", bbox_to_anchor=(1.0, 1.0), frameon=True)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

plot_lollipop(df_combined)

In [ ]:
#RQ3-4 data processing
df_exp3_unimoral = df_combined[df_combined["benchmark"] == "unimoral"]
df_exp3_mmlupro = df_combined[df_combined["benchmark"] == "mmlupro"]
df_exp3_math500 = df_combined[df_combined["benchmark"] == "math500"]

In [ ]:
#RQ3-4 plotting
def plot_reasoning_pca_kde(df, title):
    reasoning_cols = ["Backtracking", "Backward chaining", "Retrieval", "Verification",
                      "Subgoal setting", "Reframing", "Calculation", "Logistical Reasoning"]
    embedding_matrix = df[reasoning_cols].values
    pca = PCA(n_components=2)
    embedding_2d = pca.fit_transform(embedding_matrix)
    df_vis = pd.DataFrame(embedding_2d, columns=["x", "y"])
    df_vis["type"] = df["type"].values
    df_vis["avg_acc"] = df["avg_acc"].values
    unique_types = df_vis["type"].unique()

    plt.figure(figsize=(7, 5))
    color_map = {"exp1": "#1f77b4",  "exp2": "#d62728"   }
    label_map = {"exp1": "Before","exp2": "After"}
    legend_patches = []
    for t in unique_types:
        subset = df_vis[df_vis["type"] == t]
        sns.kdeplot(
            x=subset["x"],
            y=subset["y"],
            fill=True,
            color=color_map[t],
            alpha=0.5,
            levels=10,
            thresh=0.05
        )
        legend_label = label_map.get(t, t)
        legend_patches.append(Patch(facecolor=color_map[t], label=legend_label, alpha=0.5))
    
    fontsize = 18
    plt.xlabel("PCA-dim1", fontsize=fontsize)
    plt.ylabel("PCA-dim2", fontsize=fontsize)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    # plt.title(f"KDE after PCA: {title}")
    plt.legend(handles=legend_patches, title="Type", title_fontsize=fontsize, fontsize=fontsize)
    plt.tight_layout()
    plt.savefig(f"/home/yszhou/multilingual_project/experiment/analysis/figures/exp3_pca_{title.replace(' ', '_')}.pdf")
    plt.show()
    plt.close()

plot_reasoning_pca_kde(df_exp3_unimoral, title="Unimoral")
plot_reasoning_pca_kde(df_exp3_mmlupro, title="MMLUPro")
plot_reasoning_pca_kde(df_exp3_math500, title="Math500")